# HVI 빅캠 6회차 - 건축물대장 기반 HVI 품질 향상

## 주요 변경사항 (5회차 → 6회차)
- 노후도·집적도 축: B068 기반 → **건축물대장 기반**으로 대체
- 저가비율 축: B068 그대로 유지
- B068 누락 행정동의 저가비율: **자치구 평균**으로 대체
- 결측 처리: 버리는 게 아니라 해당 축 계산 시에만 제외

## 반입 데이터
- `건축물대장_빅캠반입용.csv` : 카카오 지오코딩 완료된 건축물대장 (hdong_code, total_area, total_units, 건축연도 포함)

# 0. 라이브러리 및 설정

In [ ]:
import pandas as pd
import numpy as np

ref_year = 2022
LOW_Q = 0.2
old_threshold = 30
min_building = 5
min_obs = 10

# 1. 데이터 로드

In [ ]:
def load_csv(txt_dir):
    df = pd.read_csv('E:/B068. 서울시 연립 다세대 임대 예측시세/2. 파일데이터/' + txt_dir, sep='|', dtype=str, engine='python')
    df.columns = [str(c).strip().strip('`') for c in df.columns]
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.strip('`')
    return df

def load_month(month_str):
    return (
        load_csv(f'1. 주택기본정보/jtbasicinfo_{month_str}.txt'),
        load_csv(f'3. 전세임대 예측시세/depo_hosiseinfo_{month_str}.txt'),
        load_csv(f'5. 월세임대 예측시세/rent_hosiseinfo_{month_str}.txt')
    )

months = ['202206', '202207', '202208']
all_주택, all_전세, all_월세 = [], [], []

for m in months:
    j, d, r = load_month(m)
    all_주택.append(j)
    all_전세.append(d)
    all_월세.append(r)

df_주택기본정보 = pd.concat(all_주택, ignore_index=True)
df_전세예측 = pd.concat(all_전세, ignore_index=True)
df_월세예측 = pd.concat(all_월세, ignore_index=True)

print(f'주택기본정보: {len(df_주택기본정보):,}행')
print(f'전세예측: {len(df_전세예측):,}행')
print(f'월세예측: {len(df_월세예측):,}행')

In [ ]:
# 건축물대장 반입 데이터 로드
df_bldg = pd.read_csv('건축물대장_빅캠반입용.csv', dtype={'hdong_code': str})

print(f'건축물대장: {len(df_bldg):,}행')
print(f'커버 행정동 수: {df_bldg["hdong_code"].nunique()}')
print(df_bldg.head())

# 2. B068 행정동 매핑 (5회차와 동일)

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

gdf_hdong = gpd.read_file('bnd_dong_11_2025_2Q.shp')
gdf_hdong = gdf_hdong.to_crs(epsg=4326)
gdf_hdong = gdf_hdong.rename(columns={'ADM_CD': 'hdong_code', 'ADM_NM': 'hdong_name'})

df_주택기본정보['LNG'] = pd.to_numeric(df_주택기본정보['LNG'], errors='coerce')
df_주택기본정보['LAT'] = pd.to_numeric(df_주택기본정보['LAT'], errors='coerce')

gdf_bldg_b068 = gpd.GeoDataFrame(
    df_주택기본정보,
    geometry=gpd.points_from_xy(df_주택기본정보['LNG'], df_주택기본정보['LAT']),
    crs='EPSG:4326'
)

gdf_joined = gpd.sjoin(gdf_bldg_b068, gdf_hdong[['hdong_code', 'hdong_name', 'geometry']], how='left', predicate='within')

dong_rename = {'상일1동': '상일동', '상일2동': '강일동'}
gdf_joined['hdong_name'] = gdf_joined['hdong_name'].replace(dong_rename)

print(len(gdf_bldg_b068), len(gdf_joined), gdf_joined['hdong_code'].isna().mean())

df_주택기본정보 = pd.DataFrame(gdf_joined.drop(columns='geometry'))
df_전세예측 = df_전세예측.merge(df_주택기본정보[['PKCODE1', 'hdong_code', 'hdong_name']].drop_duplicates('PKCODE1'), on='PKCODE1', how='left')
df_월세예측 = df_월세예측.merge(df_주택기본정보[['PKCODE1', 'hdong_code', 'hdong_name']].drop_duplicates('PKCODE1'), on='PKCODE1', how='left')

# 3. 축별 집계 함수 정의

In [ ]:
dong_code_col = 'hdong_code'
price_col_depo = 'DEPO_PRED'
price_col_rent = 'PRED_RENT'

df_주택기본정보['DJAREA'] = pd.to_numeric(df_주택기본정보['DJAREA'], errors='coerce')
df_전세예측[price_col_depo] = pd.to_numeric(df_전세예측[price_col_depo], errors='coerce')
df_월세예측[price_col_rent] = pd.to_numeric(df_월세예측[price_col_rent], errors='coerce')

def prepare_price_ratio(df, price_col, dong_code_col, label):
    tmp = df.copy()
    tmp = tmp[tmp[price_col].notna()].copy()
    q = tmp[price_col].quantile(LOW_Q)
    tmp[f'is_low_{label}'] = (tmp[price_col] <= q).astype(int)
    agg = (tmp.groupby(dong_code_col, dropna=False)
             .agg(n_obs=(price_col, 'size'), low_ratio=(f'is_low_{label}', 'mean'))
             .reset_index())
    agg = agg.rename(columns={'n_obs': f'n_{label}', 'low_ratio': f'low_ratio_{label}'})
    return agg, q

# 4. 축 1 - 저가 주거지 비율 (B068 기반, 5회차와 동일)

In [ ]:
depo_agg, depo_q = prepare_price_ratio(df_전세예측, price_col_depo, dong_code_col, label='depo')
rent_agg, rent_q = prepare_price_ratio(df_월세예측, price_col_rent, dong_code_col, label='rent')

print(f'전세 저가 기준: {depo_q:,.0f}만원')
print(f'월세 저가 기준: {rent_q:,.0f}만원')

# 5. 축 2 - 노후도 비율 (건축물대장 기반, 신규)

5회차에서는 B068 주택기본정보의 SYDATE를 사용했으나,  
6회차부터는 건축물대장의 `건축연도` 컬럼을 사용.

In [ ]:
# 건축물대장 기반 노후도
df_bldg['SYDATE_parsed'] = pd.to_datetime(df_bldg['SYDATE'], format='%Y%m%d', errors='coerce')
df_bldg['build_year_final'] = df_bldg['SYDATE_parsed'].dt.year

df_bldg_old = df_bldg[df_bldg['build_year_final'].notna()].copy()
df_bldg_old['building_age'] = ref_year - df_bldg_old['build_year_final'].astype(int)
df_bldg_old['is_old'] = (df_bldg_old['building_age'] >= old_threshold).astype(int)

old_agg = (df_bldg_old.groupby('hdong_code', dropna=False)
           .agg(n_building=('building_age', 'size'), old_ratio=('is_old', 'mean'))
           .reset_index())

print(f'노후도 집계 행정동 수: {len(old_agg)}')
print(old_agg.describe())

# 6. 축 3 - 집적도 (건축물대장 기반, 신규)

집합건물 표제부 특성상 대지면적이 0으로 기록된 건물은 제외.  
유효 대지면적(> 0)인 건물만 집계.

In [ ]:
df_bldg['total_area'] = pd.to_numeric(df_bldg['total_area'], errors='coerce')
df_bldg['total_units'] = pd.to_numeric(df_bldg['total_units'], errors='coerce')

# 대지면적 0 또는 결측 제외
df_bldg_density = df_bldg[(df_bldg['total_area'] > 0) & df_bldg['total_area'].notna()].copy()

print(f'집적도 계산 대상: {len(df_bldg_density):,}건 / 전체 {len(df_bldg):,}건')
print(f'제외 비율: {1 - len(df_bldg_density)/len(df_bldg):.1%}')

density_agg = (df_bldg_density.groupby('hdong_code', dropna=False)
               .agg(
                   n_building_density=('total_area', 'size'),
                   total_units=('total_units', 'sum'),
                   total_area=('total_area', 'sum')
               )
               .reset_index())

density_agg['unit_density'] = density_agg['total_units'] / density_agg['total_area']

print(f'집적도 집계 행정동 수: {len(density_agg)}')
print(density_agg['unit_density'].describe())

# 7. HVI 통합 merge

In [ ]:
# 저가비율 merge (B068 기반)
hvi = pd.merge(depo_agg, rent_agg, on='hdong_code', how='outer')

hvi['n_rent'] = hvi['n_rent'].fillna(0)
hvi['low_ratio_rent'] = hvi['low_ratio_rent'].fillna(np.nan)

def weighted_low_ratio(row):
    n_depo = row['n_depo']
    n_rent = row['n_rent']
    total = n_depo + n_rent
    if total == 0:
        return np.nan
    val = 0
    if pd.notna(row['low_ratio_depo']):
        val += n_depo * row['low_ratio_depo']
    if pd.notna(row['low_ratio_rent']):
        val += n_rent * row['low_ratio_rent']
    return val / total

hvi['low_ratio_combined'] = hvi.apply(weighted_low_ratio, axis=1)

# 노후도·집적도 merge (건축물대장 기반)
hvi = pd.merge(hvi, old_agg, on='hdong_code', how='outer')
hvi = pd.merge(hvi, density_agg, on='hdong_code', how='outer')

# 행정동명 매핑
hdong_name_map = df_주택기본정보[['hdong_code', 'hdong_name']].drop_duplicates('hdong_code')
# 건축물대장 hdong_name도 보완
bldg_name_map = df_bldg[['hdong_code', 'hdong_name']].drop_duplicates('hdong_code')
hdong_name_map = pd.concat([hdong_name_map, bldg_name_map]).drop_duplicates('hdong_code')
hvi = hvi.merge(hdong_name_map, on='hdong_code', how='left')

print(f'merge 후 행정동 수: {len(hvi)}')
print('\n결측 현황:')
print(hvi[['low_ratio_combined', 'old_ratio', 'unit_density']].isna().sum())

# 8. B068 누락 행정동 저가비율 → 자치구 평균으로 대체

In [ ]:
# 자치구 코드 추출 (hdong_code 앞 4자리)
hvi['gu_code'] = hvi['hdong_code'].astype(str).str[:4]

# 자치구별 저가비율 평균
gu_mean = hvi.groupby('gu_code')['low_ratio_combined'].mean()

# B068 없는 동 확인
missing_low = hvi['low_ratio_combined'].isna().sum()
print(f'저가비율 결측 행정동: {missing_low}개')

# 자치구 평균으로 대체
hvi['low_ratio_combined'] = hvi.apply(
    lambda row: gu_mean.get(row['gu_code'], np.nan)
    if pd.isna(row['low_ratio_combined']) else row['low_ratio_combined'],
    axis=1
)
hvi['low_ratio_imputed'] = hvi['low_ratio_combined'].isna()  # 대체 여부 플래그

print(f'대체 후 결측: {hvi["low_ratio_combined"].isna().sum()}개')

# 9. 최소 표본 필터링 및 점수 산출

In [ ]:
hvi['n_total'] = hvi['n_depo'].fillna(0) + hvi['n_rent'].fillna(0)

print(f'필터링 전: {len(hvi)}개 동')

# 세 축 모두 있어야 최종 점수 산출 가능
hvi_filtered = hvi[
    hvi['low_ratio_combined'].notna() &
    hvi['old_ratio'].notna() &
    hvi['unit_density'].notna() &
    (hvi['n_building'] >= min_building)
].copy()

print(f'필터링 후: {len(hvi_filtered)}개 동')
print(f'제외된 동: {len(hvi) - len(hvi_filtered)}개')

# 백분위 점수 변환
from scipy.stats import rankdata

def percentile_score(series):
    return (rankdata(series, method='average') - 1) / (len(series) - 1) * 100

hvi_filtered['score_low'] = percentile_score(hvi_filtered['low_ratio_combined'])
hvi_filtered['score_old'] = percentile_score(hvi_filtered['old_ratio'])
hvi_filtered['score_density'] = percentile_score(hvi_filtered['unit_density'])

hvi_filtered['HVI_score'] = (hvi_filtered['score_low'] +
                              hvi_filtered['score_old'] +
                              hvi_filtered['score_density']) / 3

hvi_filtered['HVI_index'] = hvi_filtered['HVI_score'].round().astype(int)
hvi_filtered['HVI_rank'] = hvi_filtered['HVI_score'].rank(ascending=False, method='min')
hvi_filtered['HVI_grade'] = pd.qcut(
    hvi_filtered['HVI_score'], q=5,
    labels=['매우낮음', '낮음', '보통', '높음', '매우높음']
)

# 10. Sanity Check

In [ ]:
# 상위 20개 동 확인 → 노원·도봉·강북·관악 등이 나오면 정상
top20 = hvi_filtered.sort_values('HVI_rank').head(20)[
    ['hdong_name', 'old_ratio', 'low_ratio_combined', 'unit_density', 'HVI_score', 'HVI_grade']
]
print('=== HVI 상위 20개 동 ===')
print(top20.to_string())

# 결측 비율
print('\n=== 결측 비율 ===')
print(hvi_filtered[['low_ratio_combined', 'old_ratio', 'unit_density', 'HVI_score']].isna().mean())

# 점수 분포
print('\n=== HVI_score 분포 ===')
print(hvi_filtered['HVI_score'].describe())

# 등급별 동 수
print('\n=== 등급별 동 수 ===')
print(hvi_filtered['HVI_grade'].value_counts())

# 자치구 평균으로 대체된 동 확인
print('\n=== 저가비율 자치구 평균 대체된 동 ===')
imputed = hvi_filtered[hvi_filtered['low_ratio_imputed'] == True][['hdong_name', 'gu_code', 'low_ratio_combined']]
print(f'대체된 동 수: {len(imputed)}개')
print(imputed.to_string())

# 11. 5회차 HVI와 비교

In [ ]:
# 5회차 결과 로드 (비교용)
# hvi_5th = pd.read_csv('B068_HVI.csv', dtype={'hdong_code': str})

# 행정동 수 비교
print(f'6회차 분석 행정동 수: {len(hvi_filtered)}')
# print(f'5회차 분석 행정동 수: {len(hvi_5th)}')

# 순위 변동 상위 10개 확인 (5회차 로드 후 주석 해제)
# compare = hvi_filtered[['hdong_code', 'hdong_name', 'HVI_rank']].merge(
#     hvi_5th[['hdong_code', 'HVI_rank']].rename(columns={'HVI_rank': 'HVI_rank_5th'}),
#     on='hdong_code', how='inner'
# )
# compare['rank_diff'] = compare['HVI_rank_5th'] - compare['HVI_rank']
# print(compare.sort_values('rank_diff', key=abs, ascending=False).head(10).to_string())

# 12. 최종 저장

In [ ]:
export_cols = [
    'hdong_code', 'hdong_name',
    'low_ratio_combined', 'low_ratio_imputed',
    'old_ratio', 'unit_density',
    'score_low', 'score_old', 'score_density',
    'HVI_score', 'HVI_index', 'HVI_rank', 'HVI_grade'
]

hvi_final = hvi_filtered[export_cols].copy()
hvi_final.to_csv('B068_HVI_v2.csv', index=False, encoding='utf-8-sig')

print(f'저장 완료: {len(hvi_final)}개 행정동')
print(hvi_final.head())